# Split

In [1]:
import set_env
from d2py.utils.file import mkdir
root_dir = ".temp"
mkdir(f"{root_dir}/logs")

In [35]:
import torch
from torch.nn import functional as F
from torch import nn
from torch.onnx import OperatorExportTypes, utils

class DFL(nn.Module):
    """
    Integral module of Distribution Focal Loss (DFL).
    Proposed in Generalized Focal Loss https://ieeexplore.ieee.org/document/9792391
    """

    def __init__(self, c1=16):
        """Initialize a convolutional layer with a given number of input channels."""
        super().__init__()
        self.conv = nn.Conv2d(c1, 1, 1, bias=False).requires_grad_(False)
        x = torch.arange(c1, dtype=torch.float)
        self.conv.weight.data[:] = nn.Parameter(x.view(1, c1, 1, 1))
        self.c1 = c1

    def forward(self, x):
        """Applies a transformer layer on input tensor 'x' and returns a tensor."""
        b, c, a = x.shape  # batch, channels, anchors
        # return self.conv(x.view(b, 4, self.c1, a).transpose(2, 1).softmax(1)).view(b, 4, a)
        return self.conv(x.view(b, self.c1, 4, a).softmax(1)).view(b, 4, a)

class M(nn.Module):
    def __init__(self, nc=80, ch=(16, 256, 1024)):  # detection layer
        super().__init__()
        self.conv0 = nn.Conv2d(3, 100, 1, 1, groups=1, bias=False)
        self.dfl = DFL(c1=16)

    def forward(self, x):
        x = self.conv0(x)
        x1, x2 = torch.split(x, [64, 36], dim=1)
        x1 = torch.relu(x1)
        x2 = torch.sigmoid(x2)
        return torch.cat((x1, x2), dim=1)
    
model = M().eval()

shape = 1, 3, 48, 80
xx = torch.rand(*shape, dtype=torch.float32, requires_grad=False)
# model = torch.jit.trace(model, xx)
# 导出模型
input_name = "data"
output_name = "split"
utils.export(
    model,               # torch 模型
    xx,                         # 模型输入或者对于多个输入，使用元组
    f"{root_dir}/{output_name}.onnx",               # 模型保存的位置（可以是文件或类似文件的对象）
    export_params=True,        # 将训练后的参数权重存储在模型文件内
    opset_version=17,          # 导出模型的 ONNX 版本
    do_constant_folding=True,  # 是否执行常量折叠以进行优化
    input_names = [input_name],    # 模型的输入名称
    output_names = ['output'], # 模型的输出名称
    keep_initializers_as_inputs=True,
    # export_modules_as_functions=True,
    verbose=True,
    operator_export_type=OperatorExportTypes.ONNX_FALLTHROUGH,
    # dynamic_axes={'data' : {0 : 'batch_size'},    # 可变长度的轴
    #               'output' : {0 : 'batch_size'}}
)

Exported graph: graph(%data : Float(1, 3, 48, 80, strides=[11520, 3840, 80, 1], requires_grad=0, device=cpu),
      %conv0.weight : Float(100, 3, 1, 1, strides=[3, 1, 1, 1], requires_grad=1, device=cpu)):
  %/conv0/Conv_output_0 : Float(1, 100, 48, 80, strides=[384000, 3840, 80, 1], requires_grad=0, device=cpu) = onnx::Conv[dilations=[1, 1], group=1, kernel_shape=[1, 1], pads=[0, 0, 0, 0], strides=[1, 1], onnx_name="/conv0/Conv"](%data, %conv0.weight), scope: __main__.M::/torch.nn.modules.conv.Conv2d::conv0 # /media/pc/data/tmp/cache/conda/envs/py312x/lib/python3.12/site-packages/torch/nn/modules/conv.py:456:0
  %onnx::Split_4 : Long(2, strides=[1], device=cpu) = onnx::Constant[value= 64  36 [ CPULongType{2} ]]()
  %/Split_output_0 : Float(1, 64, 48, 80, strides=[384000, 3840, 80, 1], requires_grad=1, device=cpu), %/Split_output_1 : Float(1, 36, 48, 80, strides=[384000, 3840, 80, 1], requires_grad=1, device=cpu) = onnx::Split[axis=1, onnx_name="/Split"](%/conv0/Conv_output_0, %onnx::Sp

In [36]:
import onnx
import tvm
from tvm import relay
onnx_model = onnx.load(f"{root_dir}/{output_name}.onnx")
mod, params = relay.frontend.from_onnx(onnx_model, {"data": shape}, freeze_params=True)
mod = relay.transform.InferType()(mod)
# with tvm.transform.PassContext(opt_level=3):
#     mod = relay.quantize.prerequisite_optimize(mod, params)
# mod.show()

In [37]:
print(mod)

def @main(%data: Tensor[(1, 3, 48, 80), float32] /* ty=Tensor[(1, 3, 48, 80), float32] span=/conv0/Conv.data:0:0 */) -> Tensor[(1, 100, 48, 80), float32] {
  %0 = nn.conv2d(%data, meta[relay.Constant][0] /* ty=Tensor[(100, 3, 1, 1), float32] span=/conv0/Conv.conv0.weight:0:0 */, padding=[0, 0, 0, 0], channels=100, kernel_size=[1, 1]) /* ty=Tensor[(1, 100, 48, 80), float32] span=/conv0/Conv:0:0 */;
  %1 = split(%0, indices_or_sections=[64i64], axis=1) /* ty=(Tensor[(1, 64, 48, 80), float32], Tensor[(1, 36, 48, 80), float32]) span=/Split:0:0 */;
  %2 = %1.0 /* ty=Tensor[(1, 64, 48, 80), float32] span=/Split:0:0 */;
  %3 = %1.1 /* ty=Tensor[(1, 36, 48, 80), float32] span=/Split:0:0 */;
  %4 = nn.relu(%2) /* ty=Tensor[(1, 64, 48, 80), float32] span=/Relu:0:0 */;
  %5 = sigmoid(%3) /* ty=Tensor[(1, 36, 48, 80), float32] span=/Sigmoid:0:0 */;
  %6 = (%4, %5) /* ty=(Tensor[(1, 64, 48, 80), float32], Tensor[(1, 36, 48, 80), float32]) span=/Concat:0:0 */;
  concatenate(%6, axis=1) /* ty=Tensor[

In [38]:
from tvm.relay.testing import run_infer_type
from tvm.relay.dataflow_pattern import (
    wildcard, is_op, is_tuple,
    is_constant, is_tuple_get_item,
    DFPatternCallback,
    rewrite
)
import tvm
from tvm.relay import transform as _transform

In [39]:
class Test(DFPatternCallback):
    def __init__(self, require_type=False, rewrite_once=False):
        super().__init__(require_type=require_type, rewrite_once=rewrite_once)
        self.x = wildcard()
        self.split = is_op("split")(self.x)
        self.tuple_get_item_0 = is_tuple_get_item(self.split,0)
        self.relu = is_op("nn.relu")(self.tuple_get_item_0)
        
        self.tuple_get_item_1 = is_tuple_get_item(self.split, 1)
        self.sigmoid = is_op("sigmoid")(self.tuple_get_item_1)

        self.tuple_op = is_tuple((self.relu, self.sigmoid))
        self.cat = is_op("concatenate")(self.tuple_op)
        self.pattern = self.cat

    def callback(self, pre, post, node_map):
        # x = node_map[self.x][0]
        # split = node_map.get(self.tuple_get_item, [])
        # split_lenght = int(_transform.InferTypeLocal(split[0]).shape[1])
        print("DDD")
        return post

In [40]:
# expr = rewrite(Test(), mod["main"])
pat = Test().pattern
expr = pat.partition(mod["main"])

In [41]:
print(expr)

fn (%data: Tensor[(1, 3, 48, 80), float32] /* ty=Tensor[(1, 3, 48, 80), float32] span=/conv0/Conv.data:0:0 */) -> Tensor[(1, 100, 48, 80), float32] {
  %6 = nn.conv2d(%data, meta[relay.Constant][0] /* ty=Tensor[(100, 3, 1, 1), float32] span=/conv0/Conv.conv0.weight:0:0 */, padding=[0, 0, 0, 0], channels=100, kernel_size=[1, 1]) /* ty=Tensor[(1, 100, 48, 80), float32] span=/conv0/Conv:0:0 */;
  %7 = fn (%FunctionVar_0_0, PartitionedFromPattern="split_TupleGetItem0_nn.relu_TupleGetItem1_sigmoid_Tuple_concatenate_") {
    %0 = split(%FunctionVar_0_0, indices_or_sections=[64i64], axis=1) /* ty=(Tensor[(1, 64, 48, 80), float32], Tensor[(1, 36, 48, 80), float32]) span=/Split:0:0 */;
    %1 = %0.0 /* ty=Tensor[(1, 64, 48, 80), float32] span=/Split:0:0 */;
    %2 = %0.1 /* ty=Tensor[(1, 36, 48, 80), float32] span=/Split:0:0 */;
    %3 = nn.relu(%1) /* ty=Tensor[(1, 64, 48, 80), float32] span=/Relu:0:0 */;
    %4 = sigmoid(%2) /* ty=Tensor[(1, 36, 48, 80), float32] span=/Sigmoid:0:0 */;
    %5 